In [3]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict

In [4]:
class State(TypedDict):
    topic: str
    message : str

In [5]:
from langchain_groq import ChatGroq

In [8]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [10]:
from langchain_core.messages import HumanMessage, SystemMessage


def writeAgent(state:State):
    topic = state['topic']

    response = llm.invoke(
        [
            SystemMessage(content=f"Write a blog post about given topic in 10 lines only"),
            HumanMessage(content=f"{topic} on this Topic")
        ]
    )
    
    return {"message": response.content}

In [13]:
graph = StateGraph(State)
graph.add_node("write",writeAgent)

graph.add_edge(START,"write")
graph.add_edge("write",END)

app = graph.compile()

In [19]:
config = {"configurable": {"thread_id": "1"}}
result = app.invoke({"topic":"Hare Krishna"},config=config)

In [ ]:
result["message"]


'The Hare Krishna movement is a spiritual practice.\nIt originated in India and spread globally.\nThe movement is based on Hindu scriptures, the Bhagavad Gita.\nDevotees chant the Hare Krishna mantra to find inner peace.\nThe mantra is believed to have spiritual power.\nChanting is often accompanied by music and dance.\nThe movement emphasizes the importance of love and compassion.\nIt was popularized by A.C. Bhaktivedanta Swami Prabhupada.\nThe Hare Krishna movement has a significant following worldwide.\nIt promotes a simple and spiritual way of living.'

In [20]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
graph.invoke({"foo": "", "bar":[]}, config)

{'foo': 'b', 'bar': ['a', 'b']}

In [21]:
graph.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c359-5923-681f-8002-0d7b63dfc2d3'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-30T16:41:59.573916+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c359-5912-67d1-8001-182458de851d'}}, tasks=(), interrupts=())

In [22]:
config = {"configurable": {"thread_id": "1"}}
list(graph.get_state_history(config))


[StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c359-5923-681f-8002-0d7b63dfc2d3'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-30T16:41:59.573916+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c359-5912-67d1-8001-182458de851d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c359-5912-67d1-8001-182458de851d'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-30T16:41:59.566945+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c359-5904-6237-8000-41a875e833e3'}}, tasks=(PregelTask(id='5a8e53a3-77e7-16e3-b37c-36b784e3fb68', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts

In [23]:
stream = graph.stream_events({
    "messages": [{"role": "user", "content": "What is 42 * 17?"}],
}, version="v3")

for message in stream.messages:
    for token in message.text:
        print(token, end="", flush=True)

final_state = stream.output

f:\GenAiProjects\InkSmith-AI\backend\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
f:\GenAiProjects\InkSmith-AI\backend\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


KeyError: 'thread_id'